In [2]:
import pandas as pd
import matplotlib.pyplot as plt

Matplotlib is building the font cache; this may take a moment.


In [5]:
data = pd.read_csv("./data/Food_Delivery_Times.csv")
data.head(10)


,Order_ID,Distance_km,Weather,Traffic_Level,Time_of_Day,Vehicle_Type,Preparation_Time_min,Courier_Experience_yrs,Delivery_Time_min
0,522,7.93,Windy,Low,Afternoon,Scooter,12,1.0,43
1,738,16.42,Clear,Medium,Evening,Bike,20,2.0,84
2,741,9.52,Foggy,Low,Night,Scooter,28,1.0,59
3,661,7.44,Rainy,Medium,Afternoon,Scooter,5,1.0,37
4,412,19.03,Clear,Low,Morning,Bike,16,5.0,68
5,679,19.40,Clear,Low,Evening,Scooter,8,9.0,57
6,627,9.52,Clear,Low,NaN,Bike,12,1.0,49
7,514,17.39,Clear,Medium,Evening,Scooter,5,6.0,46
8,860,1.78,Snowy,Low,Evening,Car,20,6.0,35
9,137,10.62,Foggy,Low,Evening,Scooter,29,1.0,73


In [7]:
data.shape

(1000, 9)

In [8]:
data.isna().sum()

Order_ID                   0
Distance_km                0
Weather                   30
Traffic_Level             30
Time_of_Day               30
Vehicle_Type               0
Preparation_Time_min       0
Courier_Experience_yrs    30
Delivery_Time_min          0
dtype: int64

For missing values,
-> experince is numerical value and we can assume, for missing values it will be close to mean or median
-> wheather, traffic-level or time of day we can assume will be the most probable class
-> although i think we can fill traffic level with  delivery time sensing correalation but it will be biased as its our target variable

In [10]:
data['Weather'] = data['Weather'].fillna(data['Weather'].mode()[0])
data['Traffic_Level'] = data['Traffic_Level'].fillna(data['Traffic_Level'].mode()[0])
data['Time_of_Day'] = data['Time_of_Day'].fillna(data['Time_of_Day'].mode()[0])

data['Courier_Experience_yrs'] = data['Courier_Experience_yrs'].fillna(data['Courier_Experience_yrs'].median())

In [11]:
data.describe()

,Order_ID,Distance_km,Preparation_Time_min,Courier_Experience_yrs,Delivery_Time_min
count,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000
mean,500.500000,10.059970,16.982000,4.592000,56.732000
std,288.819436,5.696656,7.204553,2.871198,22.070915
min,1.000000,0.590000,5.000000,0.000000,8.000000
25%,250.750000,5.105000,11.000000,2.000000,41.000000
50%,500.500000,10.190000,17.000000,5.000000,55.500000
75%,750.250000,15.017500,23.000000,7.000000,71.000000
max,1000.000000,19.990000,29.000000,9.000000,153.000000


In [12]:
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 9 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Order_ID                1000 non-null   int64  
 1   Distance_km             1000 non-null   float64
 2   Weather                 1000 non-null   str    
 3   Traffic_Level           1000 non-null   str    
 4   Time_of_Day             1000 non-null   str    
 5   Vehicle_Type            1000 non-null   str    
 6   Preparation_Time_min    1000 non-null   int64  
 7   Courier_Experience_yrs  1000 non-null   float64
 8   Delivery_Time_min       1000 non-null   int64  
dtypes: float64(2), int64(3), str(4)
memory usage: 70.4 KB


In [13]:
data = data.drop(['Order_ID'], axis = 1)

In [14]:
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 8 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Distance_km             1000 non-null   float64
 1   Weather                 1000 non-null   str    
 2   Traffic_Level           1000 non-null   str    
 3   Time_of_Day             1000 non-null   str    
 4   Vehicle_Type            1000 non-null   str    
 5   Preparation_Time_min    1000 non-null   int64  
 6   Courier_Experience_yrs  1000 non-null   float64
 7   Delivery_Time_min       1000 non-null   int64  
dtypes: float64(2), int64(2), str(4)
memory usage: 62.6 KB


In [ ]:
# Encode string columns using shared maps (same as app/ml/encoding.py for reuse in app)
import sys, os
_project_root = os.path.abspath("..") if os.path.basename(os.getcwd()) == "notebook" else os.getcwd()
sys.path.insert(0, _project_root)
from app.ml.encoding import (
    TRAFFIC_LEVEL_MAP,
    TIME_OF_DAY_MAP,
    WEATHER_ONEHOT_ORDER,
    VEHICLE_ONEHOT_ORDER,
    DELIVERY_FEATURE_ORDER,
)

# Ordinal: simple map
data["Traffic_Level"] = data["Traffic_Level"].map(TRAFFIC_LEVEL_MAP)
data["Time_of_Day"] = data["Time_of_Day"].map(TIME_OF_DAY_MAP)

# One-hot: first category is reference (dropped)
for w in WEATHER_ONEHOT_ORDER[1:]:
    data[f"Weather_{w}"] = (data["Weather"] == w).astype(int)
data = data.drop(columns=["Weather"])
for v in VEHICLE_ONEHOT_ORDER[1:]:
    data[f"Vehicle_Type_{v}"] = (data["Vehicle_Type"] == v).astype(int)
data = data.drop(columns=["Vehicle_Type"])

# Keep column order matching app (so trained model gets same feature order)
data = data[DELIVERY_FEATURE_ORDER + ["Delivery_Time_min"]]
data.head(10)